# DINO Training Setup for Colab

This notebook sets up and runs all 4 training experiments from the `dino-acey` branch.

The dataset is downloaded from Hugging Face: https://huggingface.co/datasets/tsbpp/fall2025_deeplearning


## Step 1: Install Dependencies


In [ ]:
# Install required packages
%pip install -q torch torchvision timm Pillow numpy scikit-learn tqdm datasets


## Step 2: Clone Repository and Checkout Branch


In [ ]:
import os
from pathlib import Path

# Clone the repository (or navigate if already cloned)
repo_url = "https://github.com/bmann51/dl_project.git"
repo_dir = Path("/content/dl_project")

if repo_dir.exists() and (repo_dir / ".git").exists():
    print(f"Repository already exists at {repo_dir}")
    os.chdir(repo_dir)
    !git checkout dino-acey
    print("✓ Switched to branch 'dino-acey'")
else:
    print(f"Cloning repository to {repo_dir}...")
    !git clone {repo_url}
    os.chdir(repo_dir)
    !git checkout dino-acey
    print("✓ Repository cloned and switched to branch 'dino-acey'")

print(f"\nCurrent directory: {os.getcwd()}")
print("✓ Ready to proceed with setup!")


## Step 3: Load Dataset from Hugging Face

Load the dataset directly from Hugging Face. No extraction needed - the dataset will be cached and used directly during training.


In [ ]:
# Load dataset from Hugging Face
# The dataset will be cached and used directly during training (no extraction needed)

from datasets import load_dataset

print("Loading dataset from Hugging Face...")
print("This will download and cache the dataset (one-time operation)...")
dataset = load_dataset("tsbpp/fall2025_deeplearning", split="train")
print(f"✓ Dataset loaded: {len(dataset)} images")
print("\n✓ No extraction needed! Dataset will be used directly during training.")


## Step 4: Update Configuration File


In [ ]:
import json
from pathlib import Path

# Read current config
config_path = Path('hyperparameter_configs.json')
if not config_path.exists():
    raise FileNotFoundError(f"Config file not found: {config_path}. Make sure you're in the dl_project directory.")

with open(config_path, 'r') as f:
    config = json.load(f)

# Check if epochs are set correctly (should be 100, not 1)
epochs_were_wrong = False
for exp_name, exp_config in config['experiments'].items():
    if exp_config.get('epochs', 0) < 10:  # If epochs is less than 10, something is wrong
        print(f"⚠️  WARNING: {exp_name} has only {exp_config.get('epochs')} epochs. Setting to 100.")
        exp_config['epochs'] = 100
        epochs_were_wrong = True

if epochs_were_wrong:
    print("\n✓ Fixed epochs to 100 for all experiments")

# Use Hugging Face dataset directly (no extraction needed!)
config['shared_args']['data_path'] = 'hf://tsbpp/fall2025_deeplearning'

# Adjust num_workers for Colab (reduce if you get errors)
config['shared_args']['num_workers'] = 2  # Colab typically works better with fewer workers

# Save updated config
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("=" * 80)
print("Configuration Updated Successfully")
print("=" * 80)
print(f"Data path: {config['shared_args']['data_path']}")
print("✓ Using Hugging Face dataset directly (no extraction needed!)")
print(f"\nExperiments to run: {list(config['experiments'].keys())}")
print(f"Total epochs per experiment: {[config['experiments'][exp]['epochs'] for exp in config['experiments']]}")

# Show experiment details
print("\n" + "=" * 80)
print("Experiment Details")
print("=" * 80)
for exp_name, exp_config in config['experiments'].items():
    print(f"\n{exp_name}:")
    print(f"  Architecture: {exp_config.get('arch', 'N/A')}")
    print(f"  Batch size: {exp_config.get('batch_size', 'N/A')}")
    print(f"  Learning rate: {exp_config.get('lr', 'N/A')}")
    print(f"  Epochs: {exp_config.get('epochs', 'N/A')}")

print("\n💡 Expected time per epoch (approximate):")
print("  - baseline, high_lr: ~15-20 minutes per epoch")
print("  - large_batch: ~8-12 minutes per epoch")  
print("  - vit_base: ~30-45 minutes per epoch")
print("\n💡 Total training time: Several hours per experiment")
print("💡 Checkpoints saved every 10 epochs to ./experiments/{experiment_name}/")


## Step 5: Launch All 4 Training Experiments

**⚠️ Important:** 
- Make sure GPU is enabled (Runtime → Change runtime type → GPU)
- Training will run sequentially (one experiment after another)
- Each experiment can take several hours
- You can monitor progress in Step 6 while training runs


### Option A: Run All Experiments Sequentially (Recommended)


In [ ]:
# Run all experiments sequentially (one after another)
# This is safer for Colab as it won't overwhelm GPU memory
# Training output is saved to ./experiments/{experiment_name}/training.log

print("Starting all 4 training experiments...")
print("Training will run sequentially. Check Step 6 to monitor progress.")
print("=" * 80)

!python launch_experiments.py --mode local


In [ ]:
### Option B: Run Experiments in Parallel (Advanced - High Memory GPU Required)


# Run all 4 experiments in parallel
# WARNING: This requires significant GPU memory (16GB+). Only use if you have a high-memory GPU.
# Uncomment the line below to use this option:

# !python launch_experiments.py --mode local --max_parallel 4


In [ ]:
### Option C: Run Specific Experiments Only


# Run only specific experiments
# Uncomment and modify the line below to run specific experiments:

# !python launch_experiments.py --mode local --experiments baseline high_lr


## Step 6: Monitor Training Progress

**💡 You can run this cell in the SAME notebook while Step 5 is running!**

After starting training in Step 5, run this cell periodically (every few minutes) to see the latest progress. 
You don't need a separate notebook - just run this cell while training continues in the background.


In [ ]:
# Option A: Run all experiments sequentially (one after another)
# This is safer for Colab as it won't overwhelm GPU memory
!python launch_experiments.py --mode local


## Step 7: Check Training Results

After training completes, use this cell to view final results and checkpoints.


In [ ]:
# View final results and checkpoints for all experiments

from pathlib import Path
import json

experiments_dir = Path("./experiments")

print("=" * 80)
print("Training Results Summary")
print("=" * 80)

if experiments_dir.exists():
    for exp_dir in sorted(experiments_dir.iterdir()):
        if exp_dir.is_dir():
            print(f"\n{'='*80}")
            print(f"Experiment: {exp_dir.name}")
            print(f"{'='*80}")
            
            # Check for checkpoints
            checkpoint_files = sorted(exp_dir.glob("checkpoint_*.pth"))
            final_checkpoint = exp_dir / "final_checkpoint.pth"
            
            if checkpoint_files:
                print(f"\n📦 Checkpoints found: {len(checkpoint_files)}")
                for ckpt in checkpoint_files[-5:]:  # Show last 5
                    size_mb = ckpt.stat().st_size / (1024 * 1024)
                    print(f"   - {ckpt.name} ({size_mb:.1f} MB)")
            
            if final_checkpoint.exists():
                size_mb = final_checkpoint.stat().st_size / (1024 * 1024)
                print(f"\n✅ Final checkpoint: {final_checkpoint.name} ({size_mb:.1f} MB)")
            
            # Check for args.json
            args_file = exp_dir / "args.json"
            if args_file.exists():
                with open(args_file, 'r') as f:
                    args = json.load(f)
                print(f"\n📋 Configuration:")
                print(f"   Architecture: {args.get('arch', 'N/A')}")
                print(f"   Batch size: {args.get('batch_size', 'N/A')}")
                print(f"   Learning rate: {args.get('lr', 'N/A')}")
                print(f"   Epochs: {args.get('epochs', 'N/A')}")
            
            # Check log file
            log_file = exp_dir / "training.log"
            if log_file.exists():
                file_size = log_file.stat().st_size
                print(f"\n📄 Training log: {file_size:,} bytes")
                print(f"   Location: {log_file}")
            
            # Check for loss log CSV
            loss_log = exp_dir / "loss_log.csv"
            if loss_log.exists():
                print(f"📊 Loss log CSV: {loss_log}")
else:
    print("No experiments directory found.")


## Important Notes

Before starting training, make sure:


In [ ]:
- **✅ GPU Enabled**: Runtime → Change runtime type → GPU (T4 or better recommended)
- **✅ Repository Cloned**: Step 2 completed successfully
- **✅ Dataset Loaded**: Step 3 completed (500,000 images)
- **✅ Config Updated**: Step 4 completed with Hugging Face dataset path

**Training Details:**
- **Training Time**: Each experiment takes several hours (100 epochs each)
- **Checkpoints**: Saved every 10 epochs to `./experiments/{experiment_name}/checkpoint_*.pth`
- **Final Model**: Saved as `final_checkpoint.pth` after training completes
- **Logs**: All training output saved to `./experiments/{experiment_name}/training.log`
- **Loss Logs**: Per-iteration losses saved to `./experiments/{experiment_name}/loss_log.csv`

**Monitoring:**
- Run Step 6 periodically to see real-time progress
- All logs are saved to disk, so you can check them anytime
- Training continues even if you close the notebook (but Colab may disconnect after inactivity)

**If you encounter errors:**
- **OOM (Out of Memory)**: Reduce batch size in `hyperparameter_configs.json`
- **Slow training**: Normal - each epoch takes 15-45 minutes depending on experiment
- **Connection issues**: Colab may disconnect after ~90 minutes of inactivity
